# NumFast — интерактивный тест-драйв

Тестируем все модули фреймворка на CuPy (GPU) / CPU-режим (автоматически).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

try:
    import cupy as xp
    print(f"CuPy {xp.__version__} — GPU: {xp.cuda.runtime.getDeviceProperties(0)['name']}")
except ImportError:
    import numpy as xp
    print(f"CuPy недоступен — CPU-режим (автоматический)")

CuPy 14.1.0 — GPU: b'NVIDIA GeForce RTX 2060'


## 1. _main — ядро

Информация о версии и состоянии расширений.

In [2]:
from _main._lib.main_lib import _get_version, _get_status

print(f"Версия: {_get_version()}")
print(f"Статус: {_get_status()}")

Версия: 0.0.0-dev
Статус: {'name': 'numfast', 'version': '0.0.0-dev', 'extensions': ['_main', 'Series', 'Tables', 'Memory', 'Mods', 'Proxy']}


## 2. Series — ряды на GPU

Проверяем, что все операции возвращают `xp.ndarray` (на GPU).

In [3]:
# -- series_main --
from Series.Series import series_main, rolling_window, normalize_series

arr = series_main([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"series_main:   type={type(arr).__name__}, shape={arr.shape}, dtype={arr.dtype}")
print(f"  данные: {arr}")

series_main:   type=ndarray, shape=(5,), dtype=float64
  данные: [1. 2. 3. 4. 5.]


In [8]:
# -- rolling_window --
win = rolling_window([1, 2, 3, 4, 5], window=3)
print(f"rolling_window(3): type={type(win).__name__}, shape={win.shape}")
print(f"  окна:\n{win}")

rolling_window(3): type=ndarray, shape=(3, 3)
  окна:
[[1. 2. 3.]
 [2. 3. 4.]
 [3. 4. 5.]]


In [9]:
# -- normalize_series --
norm = normalize_series([10, 20, 30, 40, 50])
print(f"normalize_series: type={type(norm).__name__}, shape={norm.shape}")
print(f"  нормализовано: {norm}")

normalize_series: type=ndarray, shape=(5,)
  нормализовано: [0.   0.25 0.5  0.75 1.  ]


In [10]:
# -- zero CPU transfer check --
arr = series_main([5, 5, 5])
norm = normalize_series([5, 5, 5])
print(f"Константный ряд: {norm} (ожидается [0, 0, 0])")

if 'cupy' in str(type(arr)):
    print(f"\n✅ Данные на GPU:   device={arr.device}")
else:
    print(f"\n⚠️  Данные на CPU:  автоматический CPU-режим")

Константный ряд: [0. 0. 0.] (ожидается [0, 0, 0])

✅ Данные на GPU:   device=<CUDA Device 0>


## 3. Tables — таблицы на GPU

Каждая колонка — отдельный `xp.ndarray` в dict.

In [11]:
from Tables.Tables import table_main, merge_tables, aggregate_table

t = table_main({"a": [1, 2, 3], "b": [4, 5, 6], "c": [7, 8, 9]})
print("table_main:")
for k, v in t.items():
    print(f"  {k}: type={type(v).__name__}, shape={v.shape}, data={v}")

table_main:
  a: type=ndarray, shape=(3,), data=[1. 2. 3.]
  b: type=ndarray, shape=(3,), data=[4. 5. 6.]
  c: type=ndarray, shape=(3,), data=[7. 8. 9.]


In [12]:
# -- merge_tables --
merged = merge_tables(
    {"id": [1, 2, 3], "x": [10, 20, 30]},
    {"id": [1, 2, 3], "y": [100, 200, 300]},
    on="id"
)
print("merge_tables:")
for k, v in merged.items():
    print(f"  {k}: {v}")

merge_tables:
  id: [1. 2. 3.]
  x: [10. 20. 30.]
  y: [100. 200. 300.]


In [13]:
# -- aggregate_table --
agg = aggregate_table(
    {"cat": [1, 1, 2, 2, 2], "val": [10, 20, 30, 40, 50]},
    group_by="cat",
    agg="sum"
)
print("aggregate_table (sum):")
for k, v in agg.items():
    print(f"  {k}: {v}")

agg_mean = aggregate_table(
    {"cat": [1, 1, 2, 2, 2], "val": [10, 20, 30, 40, 50]},
    group_by="cat",
    agg="mean"
)
print("\naggregate_table (mean):")
for k, v in agg_mean.items():
    print(f"  {k}: {v}")

if 'cupy' in str(type(agg['val'])):
    print(f"\n✅ Данные на GPU: device={agg['val'].device}")

aggregate_table (sum):
  cat: [1. 2.]
  val: [ 30. 120.]

aggregate_table (mean):
  cat: [1. 2.]
  val: [15. 40.]

✅ Данные на GPU: device=<CUDA Device 0>


## 4. Memory — кэш с TTL

In [14]:
from Memory.Memory import cached_calculation, clear_expired, stats
import time

# Тяжёлое вычисление (кэшируем)
def heavy(n):
    print(f"  → вычисляю heavy({n})...")
    return sum(i**2 for i in range(n))

r1 = cached_calculation("heavy_1e6", lambda: heavy(10**6), ttl=5)
print(f"Первый вызов:  {r1}")

r2 = cached_calculation("heavy_1e6", lambda: heavy(10**6), ttl=5)
print(f"Повторный:     {r2} (из кэша, heavy не вызывалась)")

print(f"\nСтатистика: {stats()}")

  → вычисляю heavy(1000000)...
Первый вызов:  333332833333500000
Повторный:     333332833333500000 (из кэша, heavy не вызывалась)

Статистика: {'total_entries': 1, 'keys': ['heavy_1e6']}


In [15]:
print("До clear_expired:", stats())
clear_expired()
print("После clear_expired:", stats())

До clear_expired: {'total_entries': 1, 'keys': ['heavy_1e6']}
После clear_expired: {'total_entries': 1, 'keys': ['heavy_1e6']}


## 5. Mods — декораторы / Registry

In [16]:
from Mods.Mods import register, get_registered

@register("my_square")
def square(x):
    return x * x

@register("my_cube")
def cube(x):
    return x ** 3

print("Зарегистрированные функции:")
for name, func in get_registered().items():
    print(f"  {name} → {func.__name__}(5) = {func(5)}")

Зарегистрированные функции:
  my_square → square(5) = 25
  my_cube → cube(5) = 125


## 6. Proxy — ленивая загрузка

In [17]:
from Proxy.Proxy import load, info

math_mod = load("math")
print(f"math.sqrt(144) = {math_mod.sqrt(144)}")

json_mod = load("json")
print(f"json.dumps([1,2,3]) = {json_mod.dumps([1,2,3])}")

print(f"\nЗагруженные модули: {info()}")

math.sqrt(144) = 12.0
json.dumps([1,2,3]) = [1, 2, 3]

Загруженные модули: {'loaded_modules': ['math', 'json']}


In [18]:
# Прокси-загружаем cupy напрямую
arr_mod = load("cupy")
print(f"Модуль: {arr_mod.__name__}")
print(f"arr_mod.arange(5) = {arr_mod.arange(5)}")

Модуль: cupy
arr_mod.arange(5) = [0 1 2 3 4]


## 7. Сводный тест — всё вместе

In [19]:
print("=" * 50)
print("СВОДНЫЙ ТЕСТ NumFast".center(50))
print("=" * 50)

# GPU/CPU
print(f"\nБэкенд: {xp.__name__.upper()} {xp.__version__}")
if 'cupy' in xp.__name__:
    dev = xp.cuda.runtime.getDeviceProperties(0)
    print(f"GPU: {dev['name']}, память: {dev['totalGlobalMem'] / 1e9:.1f} GB")
else:
    print("CPU-режим (автоматический)")

# Extensions
print(f"\nРасширения ({len(get_registered())} registred):")
for n, _ in get_registered().items():
    print(f"  ✅ {n}")

print(f"\nКэш: {stats()['total_entries']} записей")
print(f"Прокси: {info()['loaded_modules']}")
print(f"Состояние: {_get_status()}")

               СВОДНЫЙ ТЕСТ NumFast               

Бэкенд: CUPY 14.1.0
GPU: b'NVIDIA GeForce RTX 2060', память: 12.9 GB

Расширения (2 registred):
  ✅ my_square
  ✅ my_cube

Кэш: 1 записей
Прокси: ['math', 'json', 'cupy']
Состояние: {'name': 'numfast', 'version': '0.0.0-dev', 'extensions': ['_main', 'Series', 'Tables', 'Memory', 'Mods', 'Proxy']}
